# Chapter 10. Duration validation and correction layers

**Learning objective.** Separate onset, offset, and duration error, preserve the automatic extractor output, and rebuild analysis measurements from reason-coded corrections.

**Estimated time:** 75–90 minutes. **Exercises:** 10.3 and 10.4. All 120 landmark records are explicitly **synthetic teaching fixtures**; error summaries do not evaluate a real aligner.

## Input and output contract

Inputs are immutable automatic measurements, independently generated reference landmarks, a validation manifest, and a correction decision table. Outputs are observation-level validation, diagnostics, a normalized correction layer, rebuilt analysis measurements, and provenance containing pre- and post-run hashes under `companion/outputs/ch10/02_duration_validation_and_corrections/`.

In [ ]:
from pathlib import Path
import json
import sys

start = Path.cwd().resolve()
for candidate in (start, *start.parents):
    if (candidate / "companion" / "data" / "MANIFEST.tsv").is_file():
        REPOSITORY_ROOT = candidate
        break
else:
    raise RuntimeError("Run from the repository root or one of its subdirectories.")
SOURCE_ROOT = REPOSITORY_ROOT / "companion" / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
DATA_ROOT = REPOSITORY_ROOT / "companion" / "data" / "ch10" / "synthetic_acoustics"
if not DATA_ROOT.exists():
    raise FileNotFoundError("Chapter 10 fixture absent. Run companion/scripts/generate_ch10_fixture.py explicitly; no data are substituted silently.")
print("DATA STATUS: SYNTHETIC_TEACHING_FIXTURE — generated signals and landmarks, not human speech.")

In [ ]:
from phonetic_research_companion.ch10 import run_duration_validation
OUTPUT = REPOSITORY_ROOT / "companion" / "outputs" / "ch10" / "02_duration_validation_and_corrections"
result = run_duration_validation(DATA_ROOT / "automatic_landmarks.tsv", DATA_ROOT / "reference_landmarks.tsv", DATA_ROOT / "duration_validation_manifest.tsv", DATA_ROOT / "correction_decisions.tsv", OUTPUT)
assert len(result["validation"]) == 120
assert result["raw_unchanged"]
assert len(result["corrections"]) == 9
assert len(result["shifted_accurate"]) >= 5
for summary in result["summaries"]:
    print(summary)
print("Shifted-boundary but accurate-duration cases:", len(result["shifted_accurate"]))

## Learner decision

Identify a token whose two boundaries are shifted while duration remains accurate. Explain why it may be usable for a duration contrast but not for synchronization. Compare contexts and duration strata, inspect every correction reason, and demonstrate that the raw automatic columns can be recovered byte-for-byte.

In [ ]:
learner_decision = {"data_status": "SYNTHETIC_TEACHING_FIXTURE", "decision_status": "undecided", "duration_use": "Learner response required.", "synchronization_use": "Learner response required.", "correction_audit": "Learner response required.", "raw_unchanged": result["raw_unchanged"], "warning": "Synthetic error does not estimate a real aligner."}
(OUTPUT / "decision_record.json").write_text(json.dumps(learner_decision, indent=2), encoding="utf-8")
required = ["duration_validation.tsv", "duration_diagnostics.html", "measurement_corrections.tsv", "analysis_measurements.tsv", "run_provenance.json", "decision_record.json"]
assert all((OUTPUT / name).exists() for name in required)
print(json.dumps(learner_decision, indent=2))